# Q2 staged Kaggle workflow: roberta_mean_seed42

Exactly three code cells: configure/check, locate/setup, launch/inspect. See KAGGLE_RUN_GUIDE.md for required inputs. Run one phase, return its archive, inspect, then continue. No test access before evaluate. Smoke outputs are separate correctness checks, never assignment results.

Full plan: 6 training runs; seeds 42 and 123; BiGRU up to 8 epochs each, mean-pooling and first-token RoBERTa up to 3 each (28 maximum epochs total). Early stopping may shorten runs. Completion does not imply a performance target was achieved.

In [1]:
# Cell 1: configuration and installed-environment checks (no training).
from pathlib import Path
import importlib.metadata, json, os, platform, shutil, subprocess, sys
PHASE = 'roberta_mean_seed42'
SOURCE_PATH = None  # Optional exact q2_source.zip OR extracted question_2 folder.
DATA = None         # prepare: optional raw folder; others: optional prepare phase ZIP/folder.
PREVIOUS = []       # Optional explicit predecessor archives. Auto-discovery requires one per phase.
RESUME_FROM = None  # Optional own same-phase FAILED archive.
GLOVE_PATH = None   # Optional attached official GloVe ZIP/txt for prepare.
INPUTS = Path('/kaggle/input')
OUTPUT = Path('/kaggle/temp/q2-artifacts')
PUBLISH = Path('/kaggle/working')
MIN_FREE_GIB = 8
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TQDM_DISABLE'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_DATASETS_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PIP_PROGRESS_BAR'] = 'off'
print('PHASE:', PHASE, '\nPython:', sys.version, '\nPlatform:', platform.platform(), flush=True)
for name in ('torch','transformers','datasets','numpy','pandas','scikit-learn'):
    try:
        print(name, importlib.metadata.version(name), flush=True)
    except importlib.metadata.PackageNotFoundError:
        print(name, 'MISSING; non-CUDA dependencies will be checked in cell 2', flush=True)
smi = shutil.which('nvidia-smi')
if smi:
    subprocess.run([smi, '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
else:
    print('nvidia-smi: not found', flush=True)
probe = '\n'.join([
    'import sys, torch',
    'phase=sys.argv[1]',
    "required=phase not in ('prepare','finalize')",
    "print('torch CUDA available:',torch.cuda.is_available(),flush=True)",
    'if required and not torch.cuda.is_available():',
    "    raise RuntimeError('GPU phase: enable a Kaggle NVIDIA GPU before execution')",
    'if not required and torch.cuda.is_available():',
    "    raise RuntimeError('CPU phase: set Accelerator=None to preserve quota')",
    'if required:',
    "    print('GPU:',torch.cuda.get_device_name(0),'visible count:',torch.cuda.device_count(),flush=True)",
    "    x=torch.ones((4,4),device='cuda')",
    '    assert float((x@x).sum().item())==64.0',
    "    print('GPU_OPERATION_PASSED; this process uses cuda:0 only',flush=True)",
])
subprocess.run([sys.executable,'-u','-c',probe,PHASE],check=True)
free=shutil.disk_usage('/kaggle/working').free/2**30
print('Free working disk GiB:',round(free,2),flush=True)
if free < MIN_FREE_GIB:
    raise RuntimeError('Insufficient free disk for artifacts plus the phase archive')


PHASE: roberta_mean_seed42 
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0] 
Platform: Linux-6.18.55+-x86_64-with-glibc2.39
torch 2.11.0+cu128
transformers 5.16.1
datasets 4.8.5
numpy 2.1.3
pandas 2.3.3
scikit-learn 1.6.1
index, name, memory.total [MiB]
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB
torch CUDA available: True
GPU: Tesla T4 visible count: 2
GPU_OPERATION_PASSED; this process uses cuda:0 only
Free working disk GiB: 19.5


In [2]:
# Cell 2: locate/copy source, install only needed packages, verify prerequisites and plan.
import zipfile
CODE = Path('/kaggle/temp/q2-code')
CODE.mkdir(parents=True,exist_ok=True)
if SOURCE_PATH:
    source = Path(SOURCE_PATH)
else:
    archives = sorted(INPUTS.rglob('q2_source.zip'))
    if len(archives)==1:
        source=archives[0]
    elif not archives:
        roots=sorted(set(p.parent for p in INPUTS.rglob('config.json')
                         if 'q2-code' not in p.parts and (p.parent/'kaggle_start.py').exists()))
        if len(roots)!=1:
            raise RuntimeError(f'Ambiguous/missing source inputs: {roots}. Set SOURCE_PATH explicitly.')
        source=roots[0]
    else:
        raise RuntimeError('Multiple source ZIPs. Set SOURCE_PATH to one exact version.')
PROJECT=CODE/'question_2'
if source.is_file():
    with zipfile.ZipFile(source) as archive:
        for name in archive.namelist():
            if not (CODE/name).resolve().is_relative_to(CODE.resolve()):
                raise ValueError('Unsafe source archive entry')
        archive.extractall(CODE)
else:
    shutil.copytree(source,PROJECT,dirs_exist_ok=True)
if not (PROJECT/'kaggle_start.py').exists():
    raise RuntimeError('Source layout invalid: expected question_2/kaggle_start.py')
required_files = ['deliver_phase.py', 'compact_results.py', 'q2/storage.py', 'q2/compatibility.py']
missing = [name for name in required_files if not (PROJECT/name).is_file()]
if missing:
    raise RuntimeError('OUTDATED_SOURCE_DATASET: missing '+', '.join(missing)+
                       '. Update the Kaggle source dataset with the current root-level q2_source.zip, '
                       'attach its latest version, and start a fresh session. Training has not started.')
package_init = (PROJECT/'q2/__init__.py').read_text(encoding='utf-8')
if 'TQDM_DISABLE' not in package_init:
    raise RuntimeError('OUTDATED_SOURCE_DATASET: update q2_source.zip to the current plain-text logging version before training.')
print('SOURCE_VERSION_CHECK_PASSED; plain-text progress enabled', flush=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0,str(PROJECT))
# Standard-library-only helper; safe before dependency installation.
from q2.process import run_streaming
run_streaming([sys.executable,'-u',PROJECT/'setup_environment.py','--install-missing'],cwd=PROJECT)
COMMAND=[sys.executable,'-u',str(PROJECT/'kaggle_start.py'),'--phase',PHASE,
         '--config',str(PROJECT/'config.json'),'--inputs',str(INPUTS),'--output',str(OUTPUT),
         '--min-free-gib',str(MIN_FREE_GIB)]
if DATA: COMMAND+=['--data',str(DATA)]
for previous in PREVIOUS: COMMAND+=['--previous',str(previous)]
if RESUME_FROM: COMMAND+=['--resume-from',str(RESUME_FROM)]
if GLOVE_PATH: COMMAND+=['--glove-path',str(GLOVE_PATH)]
def run_phase(command, log_path=None):
    try:
        return run_streaming(command,log_path=log_path,cwd=PROJECT)
    except Exception:
        failed=OUTPUT/PHASE/f'Q2_{PHASE}_FAILED.zip'
        if failed.exists():
            run_streaming([sys.executable,'-u',PROJECT/'deliver_phase.py','--archive',failed,'--destination',PUBLISH],cwd=PROJECT)
        raise
run_phase(COMMAND+['--setup-only'])
print('SETUP_CHECK_PASSED\nPHASE:',PHASE,'\nPREREQUISITES_VERIFIED',flush=True)


SOURCE_VERSION_CHECK_PASSED; plain-text progress enabled
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Preserving installed torch: 2.11.0+cu128
dependency transformers: 5.16.1 | required ==4.51.3 | CHANGE NEEDED
dependency datasets: 4.8.5 | required ==3.6.0 | CHANGE NEEDED
dependency huggingface-hub: 1.29.0 | required ==0.30.2 | CHANGE NEEDED
dependency tokenizers: 0.23.1 | required <0.22,>=0.21 | CHANGE NEEDED
dependency safetensors: 0.8.0 | required >=0.4.3 | OK
dependency numpy: 2.1.3 | required <3,>=1.26 | OK
dependency pandas: 2.3.3 | required <3,>=2.2 | OK
dependency scikit-learn: 1.6.1 | required <2,>=1.4 | OK
dependency matplotlib: 3.10.0 | required <4,>=3.8 | OK
dependency tqdm: 4.67.3 | required <5,>=4.66 | OK
dependency requests: 2.34.2 | required <3,>=2.31 | OK
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found 

In [3]:
# Cell 3: launch ONE phase, stream logs, inspect outputs, then report success.
# Use Save Version -> Save & Run All from a fresh session. Do not launch another
# complete saved run after starting this cell interactively.
run_phase(COMMAND,log_path=OUTPUT/PHASE/'notebook_launcher.log')
archive=OUTPUT/PHASE/f'Q2_{PHASE}_results.zip'
if not archive.exists():
    raise RuntimeError('Expected phase archive missing; execution is not complete')
run_streaming([sys.executable,'-u',PROJECT/'inspect_results.py',archive,'--expected-phase',PHASE,
               '--config',PROJECT/'config.json'],cwd=PROJECT)
execution=json.loads((OUTPUT/PHASE/'execution.json').read_text())
if execution['status']!='COMPLETE':
    raise RuntimeError('Execution status is not COMPLETE')
run_streaming([sys.executable,'-u',PROJECT/'deliver_phase.py','--archive',archive,'--destination',PUBLISH],cwd=PROJECT)
archive=PUBLISH/f'Q2_{PHASE}_results.zip'
print('PHASE_COMPLETE:',PHASE,'\nDOWNLOAD:',archive,'\nINSPECTION_PASSED',flush=True)
print('Return this archive for inspection before proceeding to the next phase.',flush=True)


ENVIRONMENT_CHECK_PASSED {'utc': '2026-10-29T02:56:10.740683+00:00', 'python': '3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]', 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39', 'packages': {'torch': '2.11.0+cu128', 'transformers': '4.51.3', 'datasets': '3.6.0', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scikit-learn': '1.6.1'}, 'gpu': 'Tesla T4', 'gpu_count': 2, 'cuda': '12.8', 'source_hash': '5c284bdea9b164be9eb13cfb897f1f8e56cb8ca19bb3d0342ca80362a9d42144', 'free_disk_gib': 1048.663288116455, 'nvidia_smi': {'available': True, 'returncode': 0, 'output': 'index, name, memory.total [MiB]\n0, Tesla T4, 15360 MiB\n1, Tesla T4, 15360 MiB\n'}, 'framework_cuda_available': True, 'require_gpu': True, 'gpu_operation': {'status': 'PASS', 'device': 'cuda:0', 'checksum': 3680.0}}
Multiple GPUs allocated; this implementation uses cuda:0 only.
VERIFIED_LEGACY_PREREQUISITE: unchanged preprocessing/model/training/smoke core; original fingerprint preserved
[inspect] verified 30/40 files
INSPECTION_P